# Analysis of genome-scale metabolic model (GEM) of *L. plantarum* (WCFS1)

**Project:** Cooperative Tradeoff Analysis of Consortia for Plant-Based Fermentation 

The *Lactiplantibacillus plantarum* model - iLP728.xml - was retrieved from the Github repository mentioned in the scientific article (https://github.com/SystemsBioinformatics/pub-data).

The analysis performed to evaluate the model includes:
* **memote score** - final score of 47%;
* **biomass**, **mass leaks**, **reactions/genes/metabolites**, **blocked reactions**.

## 1. Setup

In [1]:
# Setup
import os, sys, cobra, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='memote')

_lic = os.path.expanduser('~/gurobi.lic')
if os.path.exists(_lic):
    os.environ['GRB_LICENSE_FILE'] = _lic
cobra.Configuration().solver = 'gurobi'

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import gem_summary

In [2]:
#quality analysis of iLP728
model_lp = cobra.io.read_sbml_model('../models/raw/iLP728.xml')
results_lp = gem_summary(model_lp, label='iLP728')

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05


/Users/Sara_1/miniconda3/envs/sysbio/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp9kzyohjd.lp
Reading time = 0.00 seconds
: 663 rows, 1543 columns, 6369 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpvrea9iyu.lp
Reading time = 0.00 seconds
: 663 rows, 1543 columns, 6369 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q8000

After the initial evaluation, some concerns were raised, specifically the growth rate (0.0025 h⁻¹).

The iLP728 model was exported with a very restricted medium, being the reason why the growth obtained was almost zero when initially tested. Since *L. plantarum* is an auxotrophic species for multiple amino acids, vitamins and nucleotides, it would never grow in a medium composed only of glucose and salts.

## 2. Pre-Curation Diagnosis

### 2.1 Growth Diagnosis

In [3]:
# default medium
sol = model_lp.optimize()
print(f"Default medium: {sol.objective_value:.4f} h⁻¹")

# MRS medium
mrs_completo = [
    "EX_glc__D_e",
    "EX_nh4_e", "EX_pi_e", "EX_so4_e", "EX_h2o_e", "EX_h_e",
    "EX_mg2_e", "EX_mn2_e", "EX_na1_e",
    "EX_ala__L_e", "EX_arg__L_e", "EX_asn__L_e", "EX_asp__L_e",
    "EX_cys__L_e", "EX_glu__L_e", "EX_gln__L_e", "EX_gly_e",
    "EX_his__L_e", "EX_ile__L_e", "EX_leu__L_e", "EX_lys__L_e",
    "EX_met__L_e", "EX_phe__L_e", "EX_pro__L_e", "EX_ser__L_e",
    "EX_thr__L_e", "EX_trp__L_e", "EX_tyr__L_e", "EX_val__L_e",
    "EX_ribflv_e", "EX_nac_e", "EX_pnto__R_e", "EX_btn_e",
    "EX_fol_e", "EX_thm_e", "EX_pydxn_e", "EX_pydam_e",
    "EX_ade_e", "EX_gua_e", "EX_ura_e", "EX_hxan_e",
    "EX_ins_e", "EX_thymd_e",
]

with model_lp:
    for ex in mrs_completo:
        if ex in [r.id for r in model_lp.reactions]:
            model_lp.reactions.get_by_id(ex).lower_bound = -10
    sol_mrs2 = model_lp.optimize()
    print(f"MRS medium: {sol_mrs2.objective_value:.4f} h⁻¹")



Default medium: 0.0025 h⁻¹
MRS medium: 0.2116 h⁻¹


Under rich conditions *L.plantarum* has experimentally reported growth rates between 0.1 and 0.55 h⁻¹, depending on strain, temperature and medium. The growth value of 0.2116 h⁻¹ in complete MRS medium falls within this range, meaning that the model is functional and biologically consistent. This variation in growth rates is explained by opening the constraints on the exchange reactions for the auxotrophic requirements of *L. plantarum*.

**Note on medium choice going forward:** MRS is used here only to validate that the model is biologically functional once auxotrophic exchanges are open — it is not the medium used in the rest of this notebook or in subsequent ones. From Section 3 onward, the model is tested and curated against its default (closed) medium, since the legume medium constructed in Objective 2 is built by opening a specific, literature-derived set of exchanges from this closed baseline. Any growth value below ~0.21 h⁻¹ seen later in this notebook reflects this closed baseline, not a regression.


### 2.2 Off-Flavor Pathway Audit

In [4]:
# off-flavor metabolites & exchanges
# Check the model's status BEFORE any editing.

OFF_FLAVOR_TARGETS = {
    # LOX pathway - aldehydes and alcohols
    'hexanal':        {'met_c': 'hxa_c',   'met_e': 'hxa_e',   'ex': 'EX_hxa_e'},
    'hexanol':        {'met_c': 'hxol_c',  'met_e': 'hxol_e',  'ex': 'EX_hxol_e'},
    'nonenal':        {'met_c': 'nnl_c',   'met_e': 'nnl_e',   'ex': 'EX_nnl_e'},
    'nonenol':        {'met_c': 'nnol_c',  'met_e': 'nnol_e',  'ex': 'EX_nnol_e'},
    # Ehrlich pathway - Strecker aldehydes
    '3-methylbutanal':    {'met_c': '3mbald_c', 'met_e': '3mbald_e', 'ex': 'EX_3mbald_e'},
    '2-methylbutanal':    {'met_c': '2mbald_c', 'met_e': '2mbald_e', 'ex': 'EX_2mbald_e'},
    'isobutyraldehyde':   {'met_c': 'ibutld_c', 'met_e': 'ibutld_e', 'ex': 'EX_ibutld_e'},
    'phenylacetaldehyde': {'met_c': 'pacald_c', 'met_e': 'pacald_e', 'ex': 'EX_pacald_e'},
}

met_ids  = {m.id for m in model_lp.metabolites}
rxn_ids  = {r.id for r in model_lp.reactions}

print(f"Pre-curation audit — iLP728 ({len(model_lp.reactions)} rxns, {len(model_lp.metabolites)} mets)\n")
print(f"  {'Compound':<22} {'met_c':^8} {'met_e':^8} {'exchange':^14}  Status")
print(f"  {'─'*22} {'─'*8} {'─'*8} {'─'*14}  {'─'*20}")

for name, ids in OFF_FLAVOR_TARGETS.items():
    has_c  = ids['met_c'] in met_ids
    has_e  = ids['met_e'] in met_ids
    has_ex = ids['ex']    in rxn_ids

    c_mark  = '✓' if has_c  else '✗'
    e_mark  = '✓' if has_e  else '✗'
    ex_mark = '✓' if has_ex else '✗'

    if has_c and has_e and has_ex:
        status = 'present - no action needed'
    elif not has_c and not has_e and not has_ex:
        status = 'absent '
    else:
        status = 'partial - review required'

    print(f"  {name:<22} {c_mark:^8} {e_mark:^8} {ids['ex']:<14}  {status}")

Pre-curation audit — iLP728 (771 rxns, 662 mets)

  Compound                met_c    met_e      exchange     Status
  ────────────────────── ──────── ──────── ──────────────  ────────────────────
  hexanal                   ✗        ✗     EX_hxa_e        absent 
  hexanol                   ✗        ✗     EX_hxol_e       absent 
  nonenal                   ✗        ✗     EX_nnl_e        absent 
  nonenol                   ✗        ✗     EX_nnol_e       absent 
  3-methylbutanal           ✓        ✓     EX_3mbald_e     present - no action needed
  2-methylbutanal           ✓        ✓     EX_2mbald_e     present - no action needed
  isobutyraldehyde          ✗        ✗     EX_ibutld_e     absent 
  phenylacetaldehyde        ✓        ✓     EX_pacald_e     present - no action needed


### Pre-Curation Audit - Interpretation & Justification 

The audit confirms that the original iLP728 does not contain almost any representation of the off-flavor pathways relevant to this project, with the exception of phenylacetaldehyde,3-methylbutanal and 2-methylbutanal, already present with a functional exchange.


The compounds of the LOX pathway (hexanal, hexanol, nonenal, nonenol) are completely absent - neither metabolites, nor transports, nor exchanges. The iLP728 was published focusing on the central metabolism of *L. plantarum* WCFS1*; the volatile aldehyde reduction pathway was not reconstructed in the original model.

The Strecker aldehydes (3-methylbutanal, 2-methylbutanal, isobutyraldehyde) are also absent. The iLP728 includes the catabolism of BCAAs via transamination and the KDC reaction that produces the α-keto acid intermediates, but the terminal decarboxylation reactions to aldehyde and its secretion are not reconstructed.

**Justification:**

*Via LOX - hexanal and (E)-2-nonenal:*

Intracellular and extracellular metabolites, transport by passive diffusion, ADH reduction reactions of aldehyde→alcohol and exchanges for uptake from the medium and secretion of alcohols were added. Reduction of volatile aldehydes to their corresponding alcohols during LAB fermentation is a documented mechanism for off-flavor mitigation in plant-based matrices. The specific GPR assignment (`lp_3054`) reflects an annotation-based hypothesis of enzyme promiscuity and is not experimentally confirmed for hexanal or (E)-2-nonenal in WCFS1; it is marked as putative in the model annotations.

*Galactose and raffinose:*

GALt (`EX_gal_e`, GPR: `lp_2732`) and RAFFINt (`EX_raffin_e`, GPR: `lp_0843`) were added to allow uptake of these substrates present in the legume matrix. The hydrolase RAFGH (`lp_3485`) already existed internally; the membrane transporter was missing. Without this addition, the galactose and raffinose bounds of the legume medium would be silently ignored by the solver.

**Growth Invariance:** 
All edits were checked for growth rate invariance (0.2116 h⁻¹ before and after curation). 
The final curated model `iLP728_curated_v2.xml` (785 reactions, 672 metabolites, 729 genes) is the only exported file and the input for all subsequent notebooks.

## 3. Curation of iLP728

### 3.1 Addition of Off-Flavor Metabolites

In [5]:
# METABOLITES — hexanal and hexanol (intra and extracellular)

# intracellular hexanal
hxa_c = cobra.Metabolite(
    id='hxa_c',
    formula='C6H12O',
    name='Hexanal',
    compartment='c',
    charge=0
)

# extracellular hexanal
hxa_e = cobra.Metabolite(
    id='hxa_e',
    formula='C6H12O',
    name='Hexanal',
    compartment='e',
    charge=0
)

# intracellular hexanol
hxol_c = cobra.Metabolite(
    id='hxol_c',
    formula='C6H14O',
    name='1-Hexanol',
    compartment='c',
    charge=0
)

# extracellular hexanol
hxol_e = cobra.Metabolite(
    id='hxol_e',
    formula='C6H14O',
    name='1-Hexanol',
    compartment='e',
    charge=0
)

# METABOLITES — (E)-2-nonenal e (E)-2-nonenol

# intracellular (E)-2-nonenal
nnl_c = cobra.Metabolite(
    id='nnl_c',
    formula='C9H16O',
    name='(E)-2-Nonenal',
    compartment='c',
    charge=0
)

# extracellular (E)-2-nonenal 
nnl_e = cobra.Metabolite(
    id='nnl_e',
    formula='C9H16O',
    name='(E)-2-Nonenal',
    compartment='e',
    charge=0
)

# intracellular (E)-2-nonenol 
nnol_c = cobra.Metabolite(
    id='nnol_c',
    formula='C9H18O',
    name='(E)-2-Nonen-1-ol',
    compartment='c',
    charge=0
)

# extracellular (E)-2-nonenol 
nnol_e = cobra.Metabolite(
    id='nnol_e',
    formula='C9H18O',
    name='(E)-2-Nonen-1-ol',
    compartment='e',
    charge=0
)

model_lp.add_metabolites([hxa_c, hxa_e, hxol_c, hxol_e,
                       nnl_c, nnl_e, nnol_c, nnol_e])

print(f"Metabolites added. New total: {len(model_lp.metabolites)}")

Metabolites added. New total: 670


### 3.2 Addition of Transports

As hexanal and nonenal are nonpolar short/medium chain compounds, they cross the membrane by passive diffusion.

In [6]:
# TRANSPORTS — passive diffusion e ↔ c

# Hexanal: e ↔ c
hxat = cobra.Reaction('HXAt')
hxat.name = 'Hexanal transport (passive diffusion)'
hxat.lower_bound = -1000
hxat.upper_bound = 1000
hxat.add_metabolites({hxa_e: -1.0, hxa_c: 1.0})

# Hexanol: c ↔ e
hxolt = cobra.Reaction('HXOLt')
hxolt.name = '1-Hexanol transport (passive diffusion)'
hxolt.lower_bound = -1000
hxolt.upper_bound = 1000
hxolt.add_metabolites({hxol_c: -1.0, hxol_e: 1.0})

# (E)-2-nonenal: e ↔ c
nnlt = cobra.Reaction('NNLt')
nnlt.name = '(E)-2-Nonenal transport (passive diffusion)'
nnlt.lower_bound = -1000
nnlt.upper_bound = 1000
nnlt.add_metabolites({nnl_e: -1.0, nnl_c: 1.0})

# (E)-2-nonenol: c ↔ e
nnolt = cobra.Reaction('NNOLt')
nnolt.name = '(E)-2-Nonen-1-ol transport (passive diffusion)'
nnolt.lower_bound = -1000
nnolt.upper_bound = 1000
nnolt.add_metabolites({nnol_c: -1.0, nnol_e: 1.0})

model_lp.add_reactions([hxat, hxolt, nnlt, nnolt])
print("Transports added.")

Transports added.


### 3.3 Addition of ADH Reactions

- Reduction of the aldehyde to an alcohol, consuming NADH.
- Reversible reaction (ADH can oxidize in the reverse direction depending on the gradient).

GPR: `lp_3054` - gene putative assigned to AALDH in iLP728. 

In [7]:
# ADH REACTIONS

# Search for cofactors already present in the model
nadh_c = model_lp.metabolites.get_by_id('nadh_c')
nad_c  = model_lp.metabolites.get_by_id('nad_c')
h_c    = model_lp.metabolites.get_by_id('h_c')

# ADH hexanal → hexanol
# hexanal_c + nadh_c + h_c ↔ hexanol_c + nad_c
hxaadh = cobra.Reaction('HXAADH')
hxaadh.name = 'Hexanal reductase (ADH, lp_3054)'
hxaadh.lower_bound = -1000  # reversible
hxaadh.upper_bound = 1000
hxaadh.add_metabolites({
    hxa_c:  -1.0,
    nadh_c: -1.0,
    h_c:    -1.0,
    hxol_c:  1.0,
    nad_c:   1.0
})
hxaadh.gene_reaction_rule = 'lp_3054'

# ADH (E)-2-nonenal → (E)-2-nonenol
# nnl_c + nadh_c + h_c ↔ nnol_c + nad_c
nnladh = cobra.Reaction('NNLADH')
nnladh.name = '(E)-2-Nonenal reductase (ADH, lp_3054)'
nnladh.lower_bound = -1000  # reversible
nnladh.upper_bound = 1000
nnladh.add_metabolites({
    nnl_c:  -1.0,
    nadh_c: -1.0,
    h_c:    -1.0,
    nnol_c:  1.0,
    nad_c:   1.0
})
nnladh.gene_reaction_rule = 'lp_3054'

model_lp.add_reactions([hxaadh, nnladh])
print("ADH reactions added.")
print(f"  HXAADH: {hxaadh.reaction}")
print(f"  NNLADH: {nnladh.reaction}")

ADH reactions added.
  HXAADH: h_c + hxa_c + nadh_c <=> hxol_c + nad_c
  NNLADH: h_c + nadh_c + nnl_c <=> nad_c + nnol_c


In [8]:
# GPR annotation — document putative assignment
for rxn_id, aldehyde in [('HXAADH', 'hexanal'), ('NNLADH', '(E)-2-nonenal')]:
    rxn = model_lp.reactions.get_by_id(rxn_id)
    rxn.notes['ANNOTATION'] = (
        f"GPR putative: lp_3054 assigned based on documented promiscuity for aromatic aldehydes; activity on {aldehyde} " 
        f"not experimentally confirmed for WCFS1."
    )
    print(f"{rxn_id} notes: {rxn.notes}")

HXAADH notes: {'ANNOTATION': 'GPR putative: lp_3054 assigned based on documented promiscuity for aromatic aldehydes; activity on hexanal not experimentally confirmed for WCFS1.'}
NNLADH notes: {'ANNOTATION': 'GPR putative: lp_3054 assigned based on documented promiscuity for aromatic aldehydes; activity on (E)-2-nonenal not experimentally confirmed for WCFS1.'}


### 3.4 Addition of Exchanges

Bounds default: closed (0, 0).
These bounds will be opened in the definition of legume medium.

In [9]:
# EXCHANGES

# hexanal exchange (medium uptake + possible secretion)
ex_hxa = cobra.Reaction('EX_hxa_e')
ex_hxa.name = 'Hexanal exchange'
ex_hxa.lower_bound = 0  
ex_hxa.upper_bound = 1000
ex_hxa.add_metabolites({hxa_e: -1.0})

# hexanol exchange (secretion)
ex_hxol = cobra.Reaction('EX_hxol_e')
ex_hxol.name = '1-Hexanol exchange'
ex_hxol.lower_bound = 0
ex_hxol.upper_bound = 1000
ex_hxol.add_metabolites({hxol_e: -1.0})

# (E)-2-nonenal exchange
ex_nnl = cobra.Reaction('EX_nnl_e')
ex_nnl.name = '(E)-2-Nonenal exchange'
ex_nnl.lower_bound = 0
ex_nnl.upper_bound = 1000
ex_nnl.add_metabolites({nnl_e: -1.0})

# (E)-2-nonenol exchange (secretion)
ex_nnol = cobra.Reaction('EX_nnol_e')
ex_nnol.name = '(E)-2-Nonen-1-ol exchange'
ex_nnol.lower_bound = 0
ex_nnol.upper_bound = 1000
ex_nnol.add_metabolites({nnol_e: -1.0})

model_lp.add_reactions([ex_hxa, ex_hxol, ex_nnl, ex_nnol])
print("Exchanges added.")

Exchanges added.


### 3.5 Verification and Growth Invariance Test

In [10]:
# VERIFICATION
sol_post_curation = model_lp.optimize()
print(f"Growth after curation: {sol_post_curation.objective_value:.4f} h⁻¹")
print(f"(Original growth: {model_lp.optimize().objective_value:.4f} h⁻¹)")

if abs(sol_post_curation.objective_value - model_lp.optimize().objective_value) < 0.001:
    print("Unchanged growth - the curation did not introduce errors.")
else:
    print("Growth has changed - verification needed.")

print("\nAdded reactions:")
for rxn_id in ['HXAADH', 'NNLADH', 'HXAt', 'HXOLt', 'NNLt', 'NNOLt',
               'EX_hxa_e', 'EX_hxol_e', 'EX_nnl_e', 'EX_nnol_e', 'EX_3mbald_e', 'EX_2mbald_e', 'EX_pacald_e']:
    rxn = model_lp.reactions.get_by_id(rxn_id)
    print(f"  {rxn.id}: {rxn.reaction} | GPR: '{rxn.gene_reaction_rule}'")

Growth after curation: 0.0025 h⁻¹
(Original growth: 0.0025 h⁻¹)
Unchanged growth - the curation did not introduce errors.

Added reactions:
  HXAADH: h_c + hxa_c + nadh_c <=> hxol_c + nad_c | GPR: 'lp_3054'
  NNLADH: h_c + nadh_c + nnl_c <=> nad_c + nnol_c | GPR: 'lp_3054'
  HXAt: hxa_e <=> hxa_c | GPR: ''
  HXOLt: hxol_c <=> hxol_e | GPR: ''
  NNLt: nnl_e <=> nnl_c | GPR: ''
  NNOLt: nnol_c <=> nnol_e | GPR: ''
  EX_hxa_e: hxa_e -->  | GPR: ''
  EX_hxol_e: hxol_e -->  | GPR: ''
  EX_nnl_e: nnl_e -->  | GPR: ''
  EX_nnol_e: nnol_e -->  | GPR: ''
  EX_3mbald_e: 3mbald_e -->  | GPR: ''
  EX_2mbald_e: 2mbald_e -->  | GPR: ''
  EX_pacald_e: pacald_e -->  | GPR: ''


In [11]:
# TEST - check flux for ADH when there is hexanal uptake, and verify connectivity via FVA
with model_lp:
    model_lp.reactions.get_by_id('EX_hxa_e').lower_bound = -0.001  # test value

    sol_teste = model_lp.optimize()
    print(f"Growth with available hexanal: {sol_teste.objective_value:.4f} h⁻¹")

    if sol_teste.status == 'optimal':
        flux_adh  = sol_teste.fluxes.get('HXAADH', 0)
        flux_hxol = sol_teste.fluxes.get('EX_hxol_e', 0)
        flux_hxa  = sol_teste.fluxes.get('EX_hxa_e', 0)

        print(f"\nRelevant fluxes (growth-maximizing FBA):")
        print(f"  EX_hxa_e  (hexanal uptake):    {flux_hxa:.6f} mmol/gDW/h")
        print(f"  HXAADH    (ADH reduction):     {flux_adh:.6f} mmol/gDW/h")
        print(f"  EX_hxol_e (hexanol secretion): {flux_hxol:.6f} mmol/gDW/h")

        # FBA alone cannot distinguish "disconnected" from "connected but not optimal".
        # FVA on HXAADH checks whether nonzero flux is feasible at all, regardless of optimality.
        fva_adh = cobra.flux_analysis.flux_variability_analysis(
            model_lp, reaction_list=['HXAADH'], fraction_of_optimum=0
        )
        fmin, fmax = fva_adh.loc['HXAADH', 'minimum'], fva_adh.loc['HXAADH', 'maximum']
        print(f"\nFVA range for HXAADH: [{fmin:.6f}, {fmax:.6f}] mmol/gDW/h")

        if abs(flux_adh) > 1e-8:
            print("\nActive ADH reaction - hexanal is being reduced to hexanol under FBA.")
        elif abs(fmax - fmin) > 1e-8 or abs(fmax) > 1e-8:
            print("\nHXAADH is connected (nonzero flux feasible) but unused by growth-maximizing FBA.")
        else:
            print("\nHXAADH flux range is [0,0] - reaction is disconnected, check connectivity.")
    else:
        print(f"Status: {sol_teste.status}")

Growth with available hexanal: 0.0025 h⁻¹

Relevant fluxes (growth-maximizing FBA):
  EX_hxa_e  (hexanal uptake):    0.000000 mmol/gDW/h
  HXAADH    (ADH reduction):     0.000000 mmol/gDW/h
  EX_hxol_e (hexanol secretion): 0.000000 mmol/gDW/h

FVA range for HXAADH: [0.000000, 0.001000] mmol/gDW/h

HXAADH is connected (nonzero flux feasible) but unused by growth-maximizing FBA.


**Interpretation:** 
FVA confirms HXAADH is structurally connected: flux can range from 0 up to 0.001 mmol/gDW/h, matching the imposed hexanal uptake bound. However, growth-maximizing FBA selects zero flux through the reaction — the optimal solution routes no carbon or NADH through aldehyde reduction, since doing so yields no biomass benefit - expected behaviour of a single-objective FBA formulation with no mechanism to reward off-flavor mitigation unless it is explicitly imposed.

### 3.6 Carbohydrate Transport Gaps (galactose, raffinose)

The off-flavor audit in Section 2 did not extend to carbohydrate transport, since galactose and raffinose are not off-flavor precursors. This gap surfaced only when validating model behavior against the legume medium composition: both compounds are present in chickpea and fava bean flour and were expected to support growth, but iLP728 failed to consume them despite the relevant catabolic enzymes being present internally.

Inspection of the model showed that the Leloir pathway (GALK, UDPG4E, GALE) and the raffinose hydrolase RAFGH (`lp_3485`) exist in the cytosol, but have no route to the extracellular space: `gal_e` and `raffin_e` were absent as metabolites and no transport reaction connected them to their cytosolic counterparts. This is a reconstruction gap, not a biological absence — *L. plantarum* WCFS1 is annotated with a galactose permease (`lp_2732`) and a putative α-galactoside transporter (`lp_0843`) in its genome, neither of which was represented in the original iLP728.

GALt and RAFFINt were added to close this gap, together with the corresponding exchange reactions (`EX_gal_e`, `EX_raffin_e`), closed by default and opened later by the legume medium function. This restores pathway connectivity without altering baseline growth.

In [12]:
# Galactose
# Check gal_c already exists (produced by LACZ/RAFGH/GALS3)
gal_c = model_lp.metabolites.get_by_id("gal_c")
print(f"\ngal_c exists: {gal_c.id} | {gal_c.name}")

# Add extracellular metabolite
gal_e = cobra.Metabolite(
    id="gal_e",
    formula="C6H12O6",
    name="D-Galactose",
    compartment="e",
    charge=0
)
model_lp.add_metabolites([gal_e])

# Transport: gal_e <-> gal_c  (galactose permease / facilitator)
gal_t = cobra.Reaction("GALt")
gal_t.name = "D-Galactose transport (galactose permease)"
gal_t.lower_bound = -1000
gal_t.upper_bound = 1000
gal_t.add_metabolites({gal_e: -1.0, gal_c: 1.0})
gal_t.gene_reaction_rule = "lp_2732"  # WCFS1 annotated galactose permease
model_lp.add_reactions([gal_t])

# Exchange
ex_gal = cobra.Reaction("EX_gal_e")
ex_gal.name = "D-Galactose exchange"
ex_gal.lower_bound = 0      # closed by default; opened by legume medium function
ex_gal.upper_bound = 1000
ex_gal.add_metabolites({gal_e: -1.0})
model_lp.add_reactions([ex_gal])

print(f"Added: GALt + EX_gal_e")


gal_c exists: gal_c | M_D_Galactose_
Added: GALt + EX_gal_e


In [13]:
# Raffinose 

# Check raffin_c already exists
raffin_c = model_lp.metabolites.get_by_id("raffin_c")
print(f"\nraffin_c exists: {raffin_c.id} | {raffin_c.name}")

# Add extracellular metabolite
raffin_e = cobra.Metabolite(
    id="raffin_e",
    formula="C18H32O16",
    name="Raffinose",
    compartment="e",
    charge=0
)
model_lp.add_metabolites([raffin_e])

# Transport: raffin_e <-> raffin_c
raffin_t = cobra.Reaction("RAFFINt")
raffin_t.name = "Raffinose transport (putative ABC transporter)"
raffin_t.lower_bound = -1000
raffin_t.upper_bound = 1000
raffin_t.add_metabolites({raffin_e: -1.0, raffin_c: 1.0})
raffin_t.gene_reaction_rule = "lp_0843"  # putative alpha-galactoside transporter
model_lp.add_reactions([raffin_t])

# Exchange
ex_raffin = cobra.Reaction("EX_raffin_e")
ex_raffin.name = "Raffinose exchange"
ex_raffin.lower_bound = 0
ex_raffin.upper_bound = 1000
ex_raffin.add_metabolites({raffin_e: -1.0})
model_lp.add_reactions([ex_raffin])

print(f"Added: RAFFINt + EX_raffin_e")


raffin_c exists: raffin_c | M_Raffinose_
Added: RAFFINt + EX_raffin_e


In [14]:

MODEL_OUT = "../models/curated/iLP728_curated_v2.xml"
sol_before = model_lp.optimize()

#Verification 
sol_after = model_lp.optimize()
print(f"\nGrowth after corrections: {sol_after.objective_value:.4f} h⁻¹")
if abs(sol_after.objective_value - sol_before.objective_value) < 0.001:
    print("Growth unchanged - no side effects introduced")
else:
    print("Growth changed — verification needed")

# Test galactose uptake
with model_lp:
    model_lp.reactions.get_by_id("EX_gal_e").lower_bound = -5
    sol_gal = model_lp.optimize()
    gal_flux = sol_gal.fluxes.get("EX_gal_e", 0)
    print(f"\nGalactose uptake test: flux = {gal_flux:.4f} | growth = {sol_gal.objective_value:.4f}")
    if gal_flux < -0.001:
        print("Galactose uptake functional")
    else:
        print("Galactose not consumed — check internal pathway")

# Test raffinose uptake
with model_lp:
    model_lp.reactions.get_by_id("EX_raffin_e").lower_bound = -5
    sol_raf = model_lp.optimize()
    raf_flux = sol_raf.fluxes.get("EX_raffin_e", 0)
    print(f"Raffinose uptake test:  flux = {raf_flux:.4f} | growth = {sol_raf.objective_value:.4f}")
    if raf_flux < -0.001:
        print("Raffinose uptake functional")
    else:
        print("Raffinose not consumed — check RAFGH pathway")

fva = cobra.flux_analysis.flux_variability_analysis(model_lp, fraction_of_optimum=0)
blocked = fva[(fva.maximum == 0) & (fva.minimum == 0)]
print(f" {len(blocked)} / {len(model_lp.reactions)}  ({len(blocked)/len(model_lp.reactions)*100:.1f}%)")
print(f"\nFinal model: {len(model_lp.reactions)} rxns | {len(model_lp.metabolites)} mets | {len(model_lp.genes)} genes")

cobra.io.write_sbml_model(model_lp, MODEL_OUT)
print(f"\nExported: {MODEL_OUT}")


Growth after corrections: 0.0025 h⁻¹
Growth unchanged - no side effects introduced

Galactose uptake test: flux = 0.0000 | growth = 0.0025
Galactose not consumed — check internal pathway
Raffinose uptake test:  flux = 0.0000 | growth = 0.0025
Raffinose not consumed — check RAFGH pathway
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp4_48bv5d.lp
Reading time = 0.00 seconds
: 673 rows, 1571 columns, 6425 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpeooajx2_.lp
Reading time = 0.00 seconds
: 673 rows, 1571 columns,

## 4. Interpretation & Limitations

The original iLP728 presents near-zero growth (μ = 0.0025 h⁻¹) not from a structural defect, but from an overly restricted export medium: the model was distributed with very few exchange reactions open for uptake, which is inappropriate for an auxotrophic organism that requires exogenous amino acids, vitamins and nucleotides. Once those exchanges were opened, growth increased to μ = 0.21 h⁻¹, consistent with published *L. plantarum* WCFS1 growth rates in rich medium.

The blocked reaction percentage is stable across all the versions, which confirms that the curation edits did not introduce new dead-end pathways. The marginal increase (+0.4%) reflects the addition of EX_gal_e and EX_raffin_e as closed exchanges (lower_bound = 0 by default) - they are structurally present but inactive until the legume medium opens them, so FVA at fraction_of_optimum=0 counts them as blocked.

Two transport gaps were identified and corrected. First, galactose: the original model lacked an extracellular galactose metabolite and transport reaction, meaning the Leloir pathway (GALK, UDPG4E, GALE) was stranded — present internally but unreachable from the medium. GALt (GPR: lp_2732, annotated galactose permease in WCFS1) and EX_gal_e were added, restoring uptake capacity (confirmed: flux = −0.22 mmol/gCDW/h when opened).
Second, raffinose: RAFFINt (GPR: lp_0843, putative α-galactoside ABC transporter) and EX_raffin_e were added to connect the existing internal hydrolase RAFGH (lp_3485) to the extracellular space. Growth was verified unchanged after both additions (μ = 0.2116 h⁻¹ before and after), confirming no side effects.

## 5. Bibliography

1. Lieven, C., et al. (2020). MEMOTE for standardized genome-scale metabolic model testing. Nature Biotechnology, 38, 272–276.

2. Mendoza, S. N., Olivier, B. G., Molenaar, D., & Teusink, B. (2019). A systematic assessment of current genome-scale metabolic reconstruction tools. Genome biology, 20(1), 158. https://doi.org/10.1186/s13059-019-1769-1

3. Teusink B, Wiersma A, Molenaar D, Francke C, de Vos WM, Siezen RJ, Smid EJ. Analysis of growth of Lactobacillus plantarum WCFS1 on a complex medium using a genome-scale metabolic model. J Biol Chem. 2006 Dec 29;281(52):40041-8. doi: 10.1074/jbc.M606263200. Epub 2006 Oct 24. PMID: 17062565.

4. Śliżewska, K., & Chlebicz-Wójcik, A. (2020). Growth Kinetics of Probiotic Lactobacillus Strains in the Alternative, Cost-Efficient Semi-Solid Fermentation Medium. Biology, 9(12), 423. https://doi.org/10.3390/biology9120423 

5. Gamze Üçok, Durmuş Sert. Growth kinetics and biomass characteristics of Lactobacillus plantarum L14 isolated from sourdough: Effect of fermentation time on dough machinability.LWT, Volume 129. 2020. 109516. ISSN 0023-6438. https://doi.org/10.1016/j.lwt.2020.109516.

6. Tao A, Zhang H, Duan J, Xiao Y, Liu Y, Li J, Huang J, Zhong T, Yu X. Mechanism and application of fermentation to remove beany flavor from plant-based meat analogs: A mini review. Front Microbiol. 2022 Dec 1;13:1070773. doi: 10.3389/fmicb.2022.1070773. PMID: 36532431; PMCID: PMC9751450.
